# Eksplorasi Korpus Hukum Perdata Bersih (Clean Corpus)
**Tugas Akhir**: Legal Question Answering Hukum Perdata Berbasis RAG dengan Deteksi Knowledge Poisoning  
**Korpus**: KUHPerdata (Materiil) & HIR (Acara Perdata - BAB IX & BAB XV terpilih)

Notebook ini digunakan untuk memeriksa data pasal yang telah di-chunking dan siap digunakan sebagai basis pengetahuan (*knowledge base*).

In [1]:
import json
import pandas as pd
from pathlib import Path

# Path ke korpus bersih hasil finalisasi Langkah 2 (Opsi B)
corpus_path = Path("../data/processed/legal_corpus_clean.jsonl")
manifest_path = Path("../data/manifests/trusted_corpus_manifest.json")

print(f"Mengecek file:\n- Korpus: {corpus_path.resolve()}\n- Manifest: {manifest_path.resolve()}")

Mengecek file:
- Korpus: D:\DATA\Downloads\TUGAS AKHIRR\legal-rag-perdata\data\processed\legal_corpus_clean.jsonl
- Manifest: D:\DATA\Downloads\TUGAS AKHIRR\legal-rag-perdata\data\manifests\trusted_corpus_manifest.json


In [2]:
# Load ke DataFrame Pandas
records = []
with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total seluruh pasal yang dimuat: {len(df)} pasal")
df.head()

Total seluruh pasal yang dimuat: 2276 pasal


,chunk_id,source,buku,bab,bagian,pasal,page_start,page_end,content,char_count,content_sha256,corpus_category
0,KUHPERDATA_PASAL_1,KUHPerdata,BUKU KESATU,BAB I,,1,1,1,Menikmati hak-hak kewargaan tidak tergantung p...,69,a461cf5fb12d0818f84015760b3992e1f249e49481aa6f...,NaN
1,KUHPERDATA_PASAL_2,KUHPerdata,BUKU KESATU,BAB I,,2,1,1,Anak yang ada dalam kandungan seorang perempua...,184,952e755970832a83bfd9fe176d1d3ec29ca617cd1d89b0...,NaN
2,KUHPERDATA_PASAL_3,KUHPerdata,BUKU KESATU,BAB I,,3,1,1,Tiada suatu hukuman pun yang mengakibatkan kem...,102,c8759da6f0279c842f7cb0fb3083f4e3dc3c76f5e9466f...,NaN
3,KUHPERDATA_PASAL_4,KUHPerdata,BUKU KESATU,BAB II,BAGIAN 1,4,1,1,Tanpa mengurangi ketentuan dalam Pasal 10 Kete...,360,feab6776f79b6c4cacda93bc97a4ce682d1fc9027f156c...,NaN
4,KUHPERDATA_PASAL_5,KUHPerdata,BUKU KESATU,BAB II,BAGIAN 1,5,1,2,"Presiden, setelah mendengar Mahkamah Agung men...",430,979fc44bd4f75f1ac9756865ee08c6c7307ae1b4ac248c...,NaN


### 1. Ringkasan Distribusi Dokumen

In [3]:
print("=== DISTRIBUSI SUMBER DOKUMEN ===")
print(df["source"].value_counts())
print("\n=== STATISTIK PANJANG KARAKTER (CHARS) ===")
df.groupby("source")["char_count"].describe()

=== DISTRIBUSI SUMBER DOKUMEN ===
source
KUHPerdata    2127
HIR            149
Name: count, dtype: int64

=== STATISTIK PANJANG KARAKTER (CHARS) ===


,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
HIR,149.0,601.570470,520.797209,121.0,261.0,447.0,725.0,3081.0
KUHPerdata,2127.0,367.512929,316.600993,8.0,180.5,281.0,449.0,3295.0


### 2. Memeriksa Pasal-Pasal HIR (Opsi B: BAB IX & BAB XV)

In [4]:
df_hir = df[df["source"] == "HIR"]
print(f"Total Pasal HIR Terverifikasi: {len(df_hir)}")
print("\nDistribusi Bab HIR:")
print(df_hir["bab"].value_counts())

# Tampilkan 5 pasal pertama dari BAB IX (hukum acara inti)
df_hir[df_hir["bab"].str.contains("BAB IX")][["chunk_id", "pasal", "char_count", "content"]].head()

Total Pasal HIR Terverifikasi: 149

Distribusi Bab HIR:
bab
BAB IX. HAL MENGADILI PERKARA PERDATA YANG TERMASUK WEWENANG    133
BAB XV. BERBAGAI PERATURAN                                       16
Name: count, dtype: int64


,chunk_id,pasal,char_count,content
2127,HIR_BAB_IX_PASAL_118,118,1635,(1) Tuntutan (gugatan) perdata yang pada tingk...
2128,HIR_BAB_IX_PASAL_119,119,121,Ketua pengadilan negeri berkuasa memberi nasih...
2129,HIR_BAB_IX_PASAL_120,120,209,"Jika penggugat tidak cakap menulis, maka tuntu..."
2130,HIR_BAB_IX_PASAL_120a,120a,450,(s.d.u. dg. S. 1935-102.)\n(1) Jika tuntutan ...
2131,HIR_BAB_IX_PASAL_121,121,1264,(1) Sesudah surat tuntutan yang diajukan itu a...


In [5]:
# Tampilkan pasal-pasal BAB XV yang berhasil dimasukkan (misal: psl 374 hak ingkar, psl 390 relaas pemanggilan)
df_hir[df_hir["bab"].str.contains("BAB XV")][["chunk_id", "pasal", "char_count", "content"]]

,chunk_id,pasal,char_count,content
2260,HIR_BAB_XV_PASAL_372,372,296,(1) Ketua majelis pengadilan wajib memimpin pe...
2261,HIR_BAB_XV_PASAL_373,373,367,Barang siapa mengganggu keamanan persidangan i...
2262,HIR_BAB_XV_PASAL_374,374,715,(1) Pada seorang hakim pun boleh memeriksa per...
2263,HIR_BAB_XV_PASAL_377,377,205,Jika orang Indonesia dan orang Timur Asing men...
2264,HIR_BAB_XV_PASAL_378,378,248,"Tiap orang yang dikenakan hukuman, harus pula ..."
2265,HIR_BAB_XV_PASAL_379,379,253,"Upah dan ganti rugi bagi pokrol, penasihat ata..."
2266,HIR_BAB_XV_PASAL_381,381,522,(1) Jika hakim memerintahkan orang Indonesia a...
2267,HIR_BAB_XV_PASAL_382,382,227,"Semua surat keputusan mahkamah tinggi, surat k..."
2268,HIR_BAB_XV_PASAL_383,383,207,Semua surat keputusan hakim harus tetap tersim...
2269,HIR_BAB_XV_PASAL_388,388,519,"(1) Semuajuru sita, pesuruh yang bertugas pada..."


### 3. Fungsi Pencarian Pasal Interaktif
Gunakan sel di bawah ini untuk mencari pasal berdasarkan kata kunci hukum tertentu.

In [6]:
def cari_pasal(keyword: str, source: str = None):
    filtered = df
    if source:
        filtered = filtered[filtered["source"] == source]
    matches = filtered[filtered["content"].str.contains(keyword, case=False, na=False)]
    print(f"Ditemukan {len(matches)} pasal yang memuat kata '{keyword}':")
    return matches[["chunk_id", "source", "pasal", "page_start", "content"]].head(10)

# Contoh: Cari pasal tentang 'ganti rugi' atau 'perjanjian'
cari_pasal("ganti rugi", source="KUHPerdata")

Ditemukan 51 pasal yang memuat kata 'ganti rugi':


,chunk_id,source,pasal,page_start,content
58,KUHPERDATA_PASAL_58,KUHPerdata,58,10,Janji kawin tidak menimbulkan hak untuk menunt...
82,KUHPERDATA_PASAL_82,KUHPerdata,82,14,Jika terjadi pelanggaran oleh Pegawai Catatan ...
132,KUHPERDATA_PASAL_131,KUHPerdata,131,22,"Suami atau isteri, setelah pemisahan dan pemba..."
640,KUHPERDATA_PASAL_592,KUHPerdata,592,120,Bila sebuah bengawan atau sungai mengambil jal...
642,KUHPERDATA_PASAL_594,KUHPerdata,594,120,Hak milik atas tanah yang tenggelam karena keb...
676,KUHPERDATA_PASAL_628,KUHPerdata,628,125,Pemilik mata air tidak boleh mengubah jalan al...
690,KUHPERDATA_PASAL_641,KUHPerdata,641,127,"Seorang pemilik peserta, tanpa izin dan yang l..."
700,KUHPERDATA_PASAL_651,KUHPerdata,651,128,Bila dalam memperbaiki suatu bangunan perlu di...
716,KUHPERDATA_PASAL_667,KUHPerdata,667,130,Pemilik sebidang tanah atau pekarangan yang te...
718,KUHPERDATA_PASAL_669,KUHPerdata,669,130,Bila hak atas ganti rugi tersebut pada akhir P...


### 4. Verifikasi Integritas SHA-256 (Source Verification Manifest)
Setiap chunk memiliki nilai hash unik SHA-256 yang tersimpan di manifest untuk mencegah manipulasi korpus.

In [7]:
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Manifest Name: {manifest['corpus_name']}")
print(f"Total Records: {manifest['total_chunks']}")
print("Contoh 3 catatan hash integritas pertama:")
for r in manifest["chunks"][:3]:
    print(f"- {r['chunk_id']} -> SHA256: {r['content_sha256']}")

Manifest Name: Indonesian Civil Law Clean Corpus (KUHPerdata + HIR)
Total Records: 2276
Contoh 3 catatan hash integritas pertama:
- KUHPERDATA_PASAL_1 -> SHA256: a461cf5fb12d0818f84015760b3992e1f249e49481aa6f66bddc558d9853e872
- KUHPERDATA_PASAL_2 -> SHA256: 952e755970832a83bfd9fe176d1d3ec29ca617cd1d89b0e49ce455eef87506db
- KUHPERDATA_PASAL_3 -> SHA256: c8759da6f0279c842f7cb0fb3083f4e3dc3c76f5e9466f047d76c67da6712791
